# **Practice: using Permutation importance to estimate feature importance in a linear model**

Over the lessons of this module you have studied permutation importance in detail on the theoretical side. In this notebook we suggest that you use and analyse this way of estimating feature importance in practice!

As the dataset we will use the built-in dataset from sklearn 'fetch_california_housing'. It was put together from the data of the 1990 California census. Every row in the dataset represents a separate residential district and contains information about the average characteristics of the houses in that district.

**Main characteristics:**

**Target variable (target):**

- `MedHouseVal` — the median value of the houses in the given district, expressed in hundreds of thousands of dollars.

**Features:**

- `MedInc` (Median Income): the median income in the district (in tens of thousands of dollars)

- `HouseAge` (Housing Median Age): the median age of the houses in the district (in years)
- `AveRooms` (Average Rooms per Dwelling): the average number of rooms per dwelling.
- `AveBedrms` (Average Bedrooms per Dwelling): the average number of bedrooms per dwelling.
- `Population (Population):` the size of the population in the district.
-  `AveOccup (Average Occupancy):` the average number of occupants per dwelling.
-  `Latitude (Latitude):` the geographical latitude of the district.
- `Longitude (Longitude):` the geographical longitude of the district.

The dataset contains no missing values and is a great fit for our practical task – estimating feature importance. Enjoy working through the tasks!

![jean-louis-paulin-lHwmE58fW4Y-unsplash.md.jpg](https://ucarecdn.com/246c0810-6036-4dc9-bed3-f7f7498e5790/)

# Data preprocessing

In this lesson we will apply a simple preprocessing — we will just normalise all the values. For the fun of it, let us add a couple of random features.

In [ ]:
!pip install scikit-learn==1.1.3 -q #installing the required libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

data = pd.read_csv('https://github.com/SadSabrina/explainable_AI_course/raw/refs/heads/main/data/fetch_california_housing.csv',
                   index_col=0)
data.head()

In [ ]:
X, y = data.drop('target', axis=1), data['target']

rng = np.random.RandomState(0)
bin_var = pd.Series(rng.randint(0, 1, X.shape[0]), name="rnd_bin")
num_var = pd.Series(np.arange(X.shape[0]), name="rnd_num")
X_with_rnd_feat = pd.concat((X, bin_var, num_var), axis=1)

#let us split into the training and the test sets

X_train, X_test, y_train, y_test = train_test_split(
    X_with_rnd_feat, y, random_state=42
)

# Analysing the variables

In order not to work with the features blindly, let us extract the basic information about the relationship:
- of the features with each other
- of the features with the target variable.

In [ ]:
#Let us create an instance into which we will add the target variable
train_dataset = X_train.copy()
train_dataset.insert(0, "target", y_train)
g = sns.pairplot(
    train_dataset[
        ["target", "Latitude", "AveRooms", "AveBedrms", "MedInc", 'rnd_num']
    ],
    height=1.5
)

g.fig.suptitle('Joint distributions of the continuous features',
               y=1.05,
               fontsize=13
);

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(train_dataset.drop('rnd_bin', axis=1).corr(), annot=True)

ax.set_title('Mutual correlation of the continuous features');

**Space for your conclusions:**
- ...
- ...

**Quiz 1: How many pairs of features are strongly correlated (correlation value >= 0.5)?** \
**Answer:**

**Quiz 2: Which feature, judging by the linear correlation, is the most strongly related to the target variable?** \
**Answer:**

Let us train a linear model. We will use `Lasso` from sklearn, in order to bring the possible advantages, in the form of feature selection by means of regularisation, into the model right away.


**Quiz 3: Train the model. What is the quality (the R2 metric) of the model on the test data equal to?** **Round the answer to two decimal places.**\
**Answer:**

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = Lasso(alpha=0.015)

#Train the model. What is the quality of the model on the training and the test data equal to?

lasso_model = make_pipeline(StandardScaler(), model)

# Your code here
lasso_model

print(f"Model quality on the training data: {lasso_model.score(X_train, y_train)}")
print(f"Model quality on the test data: {lasso_model.score(X_test, y_test)}")

**Quiz 4: Let us look at the coefficients of the model. Which feature turned out to be the most important?**

In [ ]:
coefs = pd.DataFrame(
    lasso_model[1].coef_, columns=["Coefficients"], index=X_train.columns
)

#Your code here
coefs.plot(kind=, figsize=(9, 7))
plt.title("Lasso model with strong regularisation")
plt.axvline(x=0, color=".5")
plt.subplots_adjust(left=0.3)

In [ ]:
coefs

**Answer:**

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.model_selection import RepeatedKFold

cv_model = cross_validate(
    lasso_model,
    X_with_rnd_feat,
    y,
    cv=RepeatedKFold(n_splits=5, n_repeats=5),
    return_estimator=True,
    n_jobs=2,
)
coefs = pd.DataFrame(
    [model[1].coef_ for model in cv_model["estimator"]],
    columns=X_with_rnd_feat.columns,
)
plt.figure(figsize=(9, 7))
sns.boxplot(data=coefs, orient="h", color="cyan", saturation=0.5)
plt.axvline(x=0, color=".5")
plt.xlabel("Importance of the coefficients")
plt.title("Importance of the coefficients and its variability")
plt.subplots_adjust(left=0.3)

**Quiz 5: Let us analyse how the spread of the features changes under cross-validation.**

**Which feature has the largest spread? (look at the standard deviation)**

In [ ]:
coefs

**Answer:**

# Feature importance by permutation

Let us recall the algorithm for computing permutation feature importance.

1. Train a model `model`. Compute its baseline quality `baseline_score`.
2. Fix a feature $f_i$. Set the number of permutation rounds `n_repeats`.
3. For every $j$ from 0 to `n_repeats`:
 - perform a random permutation of the feature $f_i$
 - train the model on the dataset where the feature $f_i$ is permuted
 - compute the quality of the new model `permuted_score`
 - estimate the change `change` equal to `baseline_score - permuted_score`
 - save the value you got
4. Represent the importance of the feature $f_i$ as the average importance over the permutations.  


**Quiz 6: Use `permutation_importance` from sklearn with the number of repeats equal to 10. Which feature has zero importance?**

In [ ]:
from sklearn.inspection import permutation_importance

perm_results = permutation_importance(scoring='neg_mean_squared_error',
                                      random_state=17)

permutation_imp_data = pd.DataFrame(perm_results.importances_mean, columns=['importances'], index=X_train.columns)
permutation_imp_data['importances_std'] = perm_results.importances_std

In [ ]:
permutation_imp_data.sort_values(by='importances_std')

**Answer:**

# What can we say about permutation importance?
**Quiz 7-8: Write down your conclusions. Choose the correct ones on Stepik.**

# Which of the conclusions from the previous step cannot be generalised to all models?

**Answer**

In [ ]:
!pip install eli5 -q

**Quiz 9: Use permutation importance from eli5. Can we say that the implementations of importance in sklearn and in eli5 give equivalent coefficients?**

In [ ]:
import eli5
from eli5.sklearn import PermutationImportance

perm = PermutationImportance(...)
eli5.show_weights(perm, feature_names = X_test.columns.tolist())